# Five-Bar Parallel Robot — Tutorial and Live Driver

The first half of this notebook is a tutorial on the inverse kinematics of this robot, with figures and an interactive widget. The second half opens the USB serial port and drives the physical Arduino. Run top to bottom on a fresh kernel.


## How the IK works

The robot has two motors fixed to a base and a single end-effector held jointly by two arms. Inverse kinematics means: given a desired end-effector position `(x, y)`, what motor angles do we need? The trick for this parallel mechanism is that the IK splits cleanly into two independent serial 2-link problems — one per arm — that happen to share the same end-effector. We can solve them one at a time using basic trigonometry.


In [ ]:
%matplotlib widget
from five_bar_tutorial import (
    plot_geometry,
    plot_single_arm,
    plot_triangle,
    plot_two_branches,
    plot_multi_target,
    interactive_solver,
    plot_workspace_map,
)


### 1. The geometry

Two motors sit symmetrically on the base at `(±BASE_D/2, 0)`. Each motor drives a proximal link of length `L1`. At the far end of each proximal link, a passive hinge (the elbow) connects to a distal link of length `L2`. The two distal links meet at a single point — the end-effector.


In [ ]:
plot_geometry()


### 2. Each arm is a 2R serial chain

Forget about the parallel structure for a moment and look at just one arm. A motor at `M`, an elbow at `E`, and the end-effector at `T`. The motor controls the proximal link's angle. The elbow is passive — it just keeps `L1` and `L2` connected. So the question for one arm is: where must the elbow be so that `|ME| = L1` and `|ET| = L2` are both satisfied?


In [ ]:
plot_single_arm(15, 50)


### 3. A triangle with three known sides

Look at the points `M`, `E`, `T`. They form a triangle whose three sides we already know: `|ME| = L1`, `|ET| = L2`, and `|MT| = r = sqrt((x − Mx)² + (y − My)²)`. With all three sides in hand, the angle at the motor follows directly from the law of cosines:

$$\cos\alpha = \frac{L_1^2 + r^2 - L_2^2}{2\,L_1\,r}$$

The motor angle in world coordinates is then $\theta = \varphi + \alpha$, where $\varphi = \mathrm{atan2}(y - M_y,\ x - M_x)$ is the direction from the motor to the target.


In [ ]:
plot_triangle(15, 50)


### 4. Two solutions per arm

Picking `α` (or equivalently, `−α`) places the elbow on either side of the motor–target line. Both elbow positions are at distance `L1` from the motor and `L2` from the target — both satisfy the constraints. The mechanism, once built, can only sit in one of the two; we have to pick a convention.


In [ ]:
plot_two_branches(15, 50)


### 5. Elbows-out working mode

The firmware uses one consistent convention: the left arm picks `+α` (its elbow sits counter-clockwise of the line of sight) and the right arm picks `−α` (clockwise). The two elbows therefore splay outward — hence "elbows-out". Each arm is solved independently with this rule, and because both solutions were computed for the same `(x, y)`, the two distal links meet at that point.


In [ ]:
plot_multi_target([(0, 60), (20, 50), (-20, 50)])


### 6. Try it yourself

Drag the sliders. The plot shows the solved linkage and prints the two motor angles and the parallel-singularity clearance. Out-of-range targets show no linkage.


In [ ]:
interactive_solver();


### 7. Workspace and singularities

Not every `(x, y)` is reachable. Beyond the outer boundary, one arm runs out of length (serial singularity). Closer to the base, the two elbows splay almost to maximum separation and the distal links become colinear (parallel singularity) — the linkage loses mechanical advantage and the motors can stall. The host-side `singularity_clearance` flags the parallel case before sending a `move`.


In [ ]:
plot_workspace_map()


---

## Drive the robot

The cells below open the USB serial port, draw a live figure synced to the physical robot, and provide `move`, `home`, `release`, and `status` helpers. Flash the firmware in `firmware/five_bar_ik/` and power the robot before running them.


In [ ]:
from five_bar_live import LiveSession
session = LiveSession("/dev/ttyUSB0")


### Demo

Each cell below is a single statement — edit the numbers and re-run.

In [ ]:
session.home()


In [ ]:
session.move(0, 60)


In [ ]:
session.move(20, 50)


In [ ]:
session.move(-20, 50)


In [ ]:
session.move(0, 40)


In [ ]:
session.home()


### Cleanup

Run before shutting down the kernel — releases the servos and closes the port.

In [ ]:
session.close()
